# LegalIR Pipeline — bản cập nhật: chiến thuật RECALL-FIRST (theo xác nhận của BTC)

BTC xác nhận: **Recall là tiêu chí xếp hạng chính** (lexicographic) — Recall cao hơn LUÔN thắng,
Precision chỉ dùng để phân định khi Recall bằng nhau tuyệt đối.

Thay đổi so với bản trước:
- **Luôn trả về đúng 5 đáp án** (giới hạn tối đa theo `scoring.py`), không dùng threshold động nữa.
- `hybrid_top_n` lấy RỘNG (candidate pool lớn) để tối đa hóa khả năng đáp án đúng lọt vào tập ứng viên.
- Thêm **Reranker** để chọn đúng 5 ứng viên tốt nhất từ candidate pool rộng — vì giờ việc "cắt còn 5"
  mới là bước quyết định Recall cuối cùng, reranker giúp chọn đúng 5 vị trí thay vì dùng thứ tự thô của Hybrid.
- Model đã chốt: `AITeamVN/Vietnamese_Embedding` (thắng rõ trong benchmark trước đó).

In [1]:
!pip install rank_bm25 pyvi sentence-transformers --break-system-packages
import numpy as np
import json, os, glob, re, gc, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from tqdm import tqdm
from pyvi import ViTokenizer
from collections import defaultdict

CACHE_DIR = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print("CUDA available:", torch.cuda.is_available())

# ---------- Tham số đã CHỐT ----------
DENSE_MODEL_NAME = "AITeamVN/Vietnamese_Embedding"
RERANKER_MODEL_NAME = "AITeamVN/Vietnamese_Reranker"
MAX_SEQ_LENGTH = 1024
RERANKER_MAX_LENGTH = 1024

ALPHA = 0.82            # trọng số Dense trong Hybrid (đã tìm từ benchmark trước)
HYBRID_TOP_N = 30        # candidate pool RỘNG trước khi rerank + cắt còn 5
NUM_ANSWERS = 5           # LUÔN LUÔN đúng 5 — giới hạn tối đa của scoring.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 53.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.3 MB/s eta 0:00:00
CUDA available: True


## Bước 1 — Chunking (title + phụ lục)

In [2]:
def extract_title(passage, search_end_pos=1000):
    """
    Trích tiêu đề văn bản. Không còn phụ thuộc vị trí 'Điều' —
    dùng search_end_pos (số ký tự đầu văn bản) làm phạm vi tìm kiếm,
    phù hợp cho cả văn bản có "Điều" lẫn không có (Công văn, QCVN, Hướng dẫn...).
    """
    preamble = passage[:search_end_pos]
    match = re.search(
        r'(THÔNG TƯ LIÊN TỊCH|THÔNG TƯ|NGHỊ ĐỊNH|QUYẾT ĐỊNH|LUẬT|CÔNG VĂN|HƯỚNG DẪN|QUY CHUẨN)\s*\n+(.*?)(?=Căn cứ|Theo đề nghị|$)',
        preamble, re.DOTALL
    )
    if match:
        title = match.group(2).replace("\r\n", " ").replace("\n", " ")
        title = re.sub(r'\s+', ' ', title).strip()
        return f"{match.group(1)}: {title}"
    return None

def chunk_document(passage, doc_id, max_words=200, overlap=30, include_appendix=True):
    dieu_positions = [m.start() for m in re.finditer(r'Điều\s+\d+\.', passage)]

    if not dieu_positions:
        # FALLBACK — dùng CHUNG hàm extract_title, chỉ đổi phạm vi tìm kiếm
        title = extract_title(passage, search_end_pos=1000)

        words = passage.split()
        chunks = []
        if len(words) <= max_words:
            text_with_title = f"{title}\n{passage.strip()}" if title else passage.strip()
            chunks.append({"doc_id": doc_id, "type": "full_doc", "text": text_with_title})
        else:
            for start in range(0, len(words), max_words - overlap):
                window = " ".join(words[start:start + max_words])
                text_with_title = f"{title}\n{window}" if title else window
                chunks.append({"doc_id": doc_id, "type": "full_doc", "text": text_with_title})
        return chunks

    # ---- Phần xử lý "Điều" + phụ lục GIỮ NGUYÊN như cũ ----
    title = extract_title(passage, search_end_pos=dieu_positions[0])
    phuluc_match = re.search(r'PHỤ LỤC', passage)
    body_end = phuluc_match.start() if phuluc_match else len(passage)
    body = passage[dieu_positions[0]:body_end]

    dieu_positions_body = [m.start() for m in re.finditer(r'Điều\s+\d+\.', body)]
    dieu_positions_body.append(len(body))

    chunks = []
    for i in range(len(dieu_positions_body) - 1):
        seg = body[dieu_positions_body[i]:dieu_positions_body[i+1]].strip()
        if seg:
            text_with_title = f"{title}\n{seg}" if title else seg
            chunks.append({"doc_id": doc_id, "type": "dieu", "text": text_with_title})

    if include_appendix and phuluc_match:
        appendix = passage[phuluc_match.start():]
        pl_positions = [m.start() for m in re.finditer(r'PHỤ LỤC[^\n]{0,40}', appendix)]
        pl_positions.append(len(appendix))
        for i in range(len(pl_positions) - 1):
            seg = appendix[pl_positions[i]:pl_positions[i+1]].strip()
            if not seg:
                continue
            words = seg.split()
            if len(words) <= max_words:
                text_with_title = f"{title}\n{seg}" if title else seg
                chunks.append({"doc_id": doc_id, "type": "phu_luc", "text": text_with_title})
            else:
                for start in range(0, len(words), max_words - overlap):
                    window = " ".join(words[start:start + max_words])
                    text_with_title = f"{title}\n{window}" if title else window
                    chunks.append({"doc_id": doc_id, "type": "phu_luc", "text": text_with_title})
    return chunks

## Bước 2 — Build corpus (cache)

In [5]:
CONTEXT_DIR = "/kaggle/input/datasets/thnhnguynchtin/legalir/selected-contexts/selected-contexts"
INCLUDE_APPENDIX = True
MAX_WORDS = 200

cfg_tag = f"{'with' if INCLUDE_APPENDIX else 'no'}app_w{MAX_WORDS}"
# corpus_cache_path = f"{CACHE_DIR}/corpus_{cfg_tag}.json"
corpus_cache_path = "/kaggle/input/datasets/thnhnguynchtin/legalir/results/cache/corpus_withapp_w200.json"

if os.path.exists(corpus_cache_path):
    print("Đang load corpus từ cache...")
    cached = json.load(open(corpus_cache_path, encoding="utf-8"))
    corpus_texts, corpus_doc_ids = cached["texts"], cached["doc_ids"]
else:
    print("Đang build corpus từ file gốc...")
    all_chunks = []
    for fpath in tqdm(sorted(glob.glob(os.path.join(CONTEXT_DIR, "*.json"))), desc="Chunking documents"):
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)
        doc_chunks = chunk_document(data["passage"], str(data["id"]),
                                     max_words=MAX_WORDS, include_appendix=INCLUDE_APPENDIX)
        all_chunks.extend(doc_chunks)

    corpus_texts = [c["text"] for c in all_chunks]
    corpus_doc_ids = [c["doc_id"] for c in all_chunks]

    json.dump({"texts": corpus_texts, "doc_ids": corpus_doc_ids},
               open(corpus_cache_path, "w", encoding="utf-8"))

print(f"Số chunk trong corpus: {len(corpus_texts)}")

Đang load corpus từ cache...
Số chunk trong corpus: 261318


## Bước 3 — Hàm tiện ích (dedupe theo doc_id, đánh giá Recall/Precision đúng công thức scoring.py)

In [6]:
def dedupe_by_docid(ranked_chunk_ids, scores, corpus_doc_ids):
    """Dedupe theo doc_id, giữ điểm CAO NHẤT trong các chunk cùng doc_id, trả về list[(doc_id, score)] đã sort."""
    best_score = {}
    for cid, score in zip(ranked_chunk_ids, scores):
        doc_id = corpus_doc_ids[cid]
        if doc_id not in best_score or score > best_score[doc_id]:
            best_score[doc_id] = score
    return sorted(best_score.items(), key=lambda x: -x[1])


def top_n_answers(sorted_docid_score_pairs, n=5):
    """Luôn trả về đúng n đáp án (nếu đủ), không dùng threshold."""
    return [doc_id for doc_id, _ in sorted_docid_score_pairs[:n]]


def precision_recall(predicted_docids, gold_docids, max_k=5):
    """Đúng công thức scoring.py của BTC: 0 điểm nếu rỗng hoặc vượt quá max_k."""
    pred_set = set(predicted_docids)
    if len(pred_set) == 0 or len(pred_set) > max_k:
        return 0.0, 0.0
    tp = len(pred_set & gold_docids)
    precision = tp / len(pred_set)
    recall = tp / len(gold_docids) if gold_docids else 0.0
    return precision, recall

## Bước 4 — Tokenize + BM25 index (cache) — dùng `underthesea` (có trong danh sách BTC đã duyệt)

In [7]:
def tokenize_doc(text: str):
    segmented_text = ViTokenizer.tokenize(str(text))
    text_clean = re.sub(r"[^\w\s]", " ", segmented_text)
    return text_clean.lower().split()

tokenize = tokenize_doc

# tokenized_cache_path = f"{CACHE_DIR}/tokenized_corpus_{cfg_tag}.json"
tokenized_cache_path = "/kaggle/input/datasets/thnhnguynchtin/legalir/results/cache/tokenized_corpus_withapp_w200.json"

if os.path.exists(tokenized_cache_path):
    print("Đang load tokenized corpus từ cache...")
    tokenized_corpus = json.load(open(tokenized_cache_path, encoding="utf-8"))
else:
    print(f"Đang tokenize {len(corpus_texts)} chunk...")
    with ProcessPoolExecutor() as executor:
        tokenized_corpus = list(
            tqdm(executor.map(tokenize_doc, corpus_texts, chunksize=50),
                 total=len(corpus_texts), desc="Tokenizing")
        )
    json.dump(tokenized_corpus, open(tokenized_cache_path, "w", encoding="utf-8"))

bm25 = BM25Okapi(tokenized_corpus)
print("Đã build xong BM25 index.")

Đang load tokenized corpus từ cache...
Đã build xong BM25 index.


## Bước 5 — Dense encode corpus (GPU, fp16, cache)

In [8]:
dense_model = SentenceTransformer(DENSE_MODEL_NAME, device='cuda')
dense_model.max_seq_length = MAX_SEQ_LENGTH
dense_model = dense_model.half()
print("Dense model dtype:", next(dense_model.parameters()).dtype)

safe_name = DENSE_MODEL_NAME.replace("/", "_")
# embeddings_cache_path = f"{CACHE_DIR}/encoded_corpus_{cfg_tag}.json"
embeddings_cache_path = "/kaggle/input/datasets/thnhnguynchtin/legalir/results/cache/encoded_corpus_withapp_w200.json"

if os.path.exists(embeddings_cache_path):
    print("Đang load corpus embeddings từ cache...")
    corpus_embeddings = torch.load(embeddings_cache_path, weights_only=False)
else:
    print("Đang encode corpus...")
    corpus_embeddings = dense_model.encode(
        corpus_texts, batch_size=256, convert_to_tensor=True,
        show_progress_bar=True, device='cuda'
    )
    torch.save(corpus_embeddings, embeddings_cache_path)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Dense model dtype: torch.float16
Đang load corpus embeddings từ cache...


## Bước 6 — Load Reranker (dùng để chọn đúng 5 ứng viên tốt nhất từ candidate pool rộng)

In [9]:
reranker = CrossEncoder(RERANKER_MODEL_NAME, max_length=RERANKER_MAX_LENGTH, device='cuda')
print("Reranker đã sẵn sàng, max_length =", RERANKER_MAX_LENGTH)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reranker đã sẵn sàng, max_length = 1024


## Bước 7 — Hàm tính điểm Hybrid (BM25 + Dense) cho 1 lô câu hỏi

In [10]:
def bm25_score_one(q):
    return bm25.get_scores(tokenize(q)).astype(np.float32)


def compute_batch_scores(q_batch, encode_batch_size=64, executor=None):
    dense_batch = util.cos_sim(
        dense_model.encode(q_batch, convert_to_tensor=True,
                            batch_size=encode_batch_size, show_progress_bar=False),
        corpus_embeddings
    )

    if executor is not None:
        bm25_batch_np = np.array(list(executor.map(bm25_score_one, q_batch)), dtype=np.float32)
    else:
        bm25_batch_np = np.array([bm25_score_one(q) for q in q_batch], dtype=np.float32)

    bm25_batch = torch.tensor(bm25_batch_np, device='cuda')
    return dense_batch, bm25_batch

In [11]:
def find_best_alpha_recall(questions, golds, alphas, hybrid_top_n=HYBRID_TOP_N,
                            num_answers=NUM_ANSWERS, encode_batch_size=64,
                            question_chunk_size=100, n_threads=8):
    n = len(questions)

    print(f"{'Alpha':<8} {'Precision':>12} {'Recall':>10}")
    best_alpha, best_recall, best_precision = None, -1, None

    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        # Tính BM25 + Dense 1 lần, dùng lại cho MỌI alpha (giống thiết kế find_best_alpha_theta cũ)
        all_bm25, all_dense = [], []
        for start in tqdm(range(0, n, question_chunk_size), desc="Computing scores"):
            end = min(start + question_chunk_size, n)
            q_batch = questions[start:end]
            dense_batch, bm25_batch = compute_batch_scores(q_batch, encode_batch_size, executor=executor)
            bm25_norm = bm25_batch / (bm25_batch.max(dim=1, keepdim=True).values + 1e-9)
            dense_norm = dense_batch / (dense_batch.max(dim=1, keepdim=True).values + 1e-9)
            all_bm25.append(bm25_norm.cpu())
            all_dense.append(dense_norm.cpu())
            del dense_batch, bm25_batch, bm25_norm, dense_norm
            torch.cuda.empty_cache(); gc.collect()

    for alpha in alphas:
        p_sum, r_sum = 0.0, 0.0
        idx = 0
        for bm25_norm, dense_norm in zip(all_bm25, all_dense):
            final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm
            _, topk_idx = torch.topk(final_scores, hybrid_top_n, dim=1)
            topk_idx_np = topk_idx.numpy()
            topk_vals_np = torch.gather(final_scores, 1, topk_idx).numpy()

            for i in range(topk_idx_np.shape[0]):
                sorted_pairs = dedupe_by_docid(topk_idx_np[i], topk_vals_np[i], corpus_doc_ids)
                pred = top_n_answers(sorted_pairs, n=num_answers)
                p, r = precision_recall(pred, golds[idx])
                p_sum += p
                r_sum += r
                idx += 1

        p_avg, r_avg = p_sum / n, r_sum / n
        print(f"{alpha:<8} {p_avg:>12.4f} {r_avg:>10.4f}")

        if r_avg > best_recall:   # <-- ưu tiên Recall tuyệt đối, đúng luật lexicographic BTC xác nhận
            best_recall, best_alpha, best_precision = r_avg, alpha, p_avg

    print(f"\n>>> Alpha tối ưu Recall: {best_alpha} (Recall={best_recall:.4f}, Precision={best_precision:.4f})")
    return best_alpha

## Bước 8 — Hàm dự đoán CUỐI CÙNG: Hybrid (candidate pool rộng) → Rerank → cắt đúng 5

In [12]:
def predict_questions_recall_first(questions, alpha=ALPHA, hybrid_top_n=HYBRID_TOP_N,
                                    num_answers=NUM_ANSWERS, use_rerank=True,
                                    encode_batch_size=64, question_chunk_size=100,
                                    n_threads=8, rerank_batch_size=256):
    """
    Trả về list[list[doc_id]] — LUÔN đúng num_answers đáp án cho mỗi câu hỏi.
    Quy trình: BM25+Dense (Hybrid) -> Rerank (nếu use_rerank=True) -> cắt còn num_answers.
    """
    n = len(questions)
    predictions = []

    print(f"Đang dự đoán cho {n} câu hỏi (alpha={alpha}, top_n={hybrid_top_n}, rerank={use_rerank})...")

    with torch.inference_mode():
        with ThreadPoolExecutor(max_workers=n_threads) as executor:
            for start in tqdm(range(0, n, question_chunk_size), desc="Predicting"):
                end = min(start + question_chunk_size, n)
                q_batch = questions[start:end]

                # 1. Tính toán hybrid scores
                dense_batch, bm25_batch = compute_batch_scores(q_batch, encode_batch_size, executor=executor)

                bm25_norm = bm25_batch / (bm25_batch.max(dim=1, keepdim=True).values + 1e-9)
                dense_norm = dense_batch / (dense_batch.max(dim=1, keepdim=True).values + 1e-9)

                final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm
                topk_vals, topk_idx = torch.topk(final_scores, hybrid_top_n, dim=1)
                
                # Chuyển dữ liệu sang CPU NumPy
                topk_idx_np = topk_idx.cpu().numpy()
                topk_vals_np = topk_vals.cpu().numpy()

                # Giải phóng VRAM ngay lập tức
                del dense_batch, bm25_batch, bm25_norm, dense_norm, final_scores, topk_vals, topk_idx
                torch.cuda.empty_cache()

                # 2. Bước Rerank
                if use_rerank:
                    all_pairs = []
                    for i in range(len(q_batch)):
                        for cid in topk_idx_np[i]:
                            all_pairs.append([q_batch[i], corpus_texts[cid]])

                    # Chạy Rerank với batch_size nhỏ hơn (32)
                    rerank_scores = reranker.predict(
                        all_pairs, 
                        batch_size=rerank_batch_size, 
                        show_progress_bar=False
                    )

                    idx_cursor = 0
                    for i in range(len(q_batch)):
                        m = len(topk_idx_np[i])
                        scores_i = rerank_scores[idx_cursor: idx_cursor + m]
                        idx_cursor += m
                        sorted_pairs = dedupe_by_docid(topk_idx_np[i], scores_i, corpus_doc_ids)
                        pred = top_n_answers(sorted_pairs, n=num_answers)
                        predictions.append(pred)

                    del all_pairs, rerank_scores
                else:
                    for i in range(len(q_batch)):
                        sorted_pairs = dedupe_by_docid(topk_idx_np[i], topk_vals_np[i], corpus_doc_ids)
                        pred = top_n_answers(sorted_pairs, n=num_answers)
                        predictions.append(pred)

                # Thu gom rác bộ nhớ CPU & GPU ở cuối mỗi batch
                del topk_idx_np, topk_vals_np
                torch.cuda.empty_cache()
                gc.collect()

    return predictions

## Bước 9 — Load tập mẫu từ train.json, kiểm tra Recall trần (oracle) theo `hybrid_top_n`

In [13]:
with open("/kaggle/input/datasets/thnhnguynchtin/legalir/train.json", encoding="utf-8") as f:
    train_data = json.load(f)

import random
random.seed(42)
SAMPLE_SIZE = 800
train_qids_all = list(train_data.keys())
sample_qids = random.sample(train_qids_all, min(SAMPLE_SIZE, len(train_qids_all)))

questions_sample = [train_data[qid]["question"] for qid in sample_qids]
golds_sample = [set(train_data[qid]["answer"]) for qid in sample_qids]

print(f"Số câu hỏi mẫu: {len(questions_sample)}")


def oracle_recall_at_n(questions, golds, alpha, top_n_list, encode_batch_size=64,
                        question_chunk_size=100, n_threads=8):
    """Kiểm tra Recall TRẦN (không rerank, không cắt 5) tại các mức hybrid_top_n khác nhau —
    xem đáp án đúng có nằm trong candidate pool hay không, TRƯỚC KHI cắt còn 5."""
    n = len(questions)
    max_top_n = max(top_n_list)

    recall_sum_per_n = {t: 0.0 for t in top_n_list}

    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        for start in tqdm(range(0, n, question_chunk_size), desc="Oracle recall"):
            end = min(start + question_chunk_size, n)
            q_batch = questions[start:end]
            gold_batch = golds[start:end]

            dense_batch, bm25_batch = compute_batch_scores(q_batch, encode_batch_size, executor=executor)
            bm25_norm = bm25_batch / (bm25_batch.max(dim=1, keepdim=True).values + 1e-9)
            dense_norm = dense_batch / (dense_batch.max(dim=1, keepdim=True).values + 1e-9)
            final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm
            _, topk_idx = torch.topk(final_scores, max_top_n, dim=1)
            topk_idx_np = topk_idx.cpu().numpy()

            for i in range(len(q_batch)):
                gold = gold_batch[i]
                seen_docids = []
                seen_set = set()
                for cid in topk_idx_np[i]:
                    doc_id = corpus_doc_ids[cid]
                    if doc_id not in seen_set:
                        seen_set.add(doc_id)
                        seen_docids.append(doc_id)

                for t in top_n_list:
                    top_t_docids = set(seen_docids[:t])
                    tp = len(top_t_docids & gold)
                    recall_sum_per_n[t] += tp / len(gold) if gold else 0

            del dense_batch, bm25_batch, bm25_norm, dense_norm
            torch.cuda.empty_cache()
            gc.collect()

    print(f"{'top_n':<8} {'Oracle Recall':>15}")
    for t in top_n_list:
        print(f"{t:<8} {recall_sum_per_n[t]/n:>15.4f}")

ALPHAS_TEST = [0.5, 0.6, 0.7, 0.75, 0.8, 0.82, 0.85, 0.9]
best_alpha = find_best_alpha_recall(questions_sample, golds_sample, ALPHAS_TEST)
oracle_recall_at_n(questions_sample, golds_sample, alpha=best_alpha, top_n_list=[5, 10, 20, 30, 50])

Số câu hỏi mẫu: 800
Alpha       Precision     Recall


Computing scores: 100%|██████████| 8/8 [21:29<00:00, 161.23s/it]


0.5            0.2024     0.9057
0.6            0.2017     0.9116
0.7            0.2033     0.9191
0.75           0.2040     0.9214
0.8            0.2034     0.9189
0.82           0.2027     0.9153
0.85           0.2009     0.9113
0.9            0.1987     0.9088

>>> Alpha tối ưu Recall: 0.75 (Recall=0.9214, Precision=0.2040)


Oracle recall: 100%|██████████| 8/8 [21:16<00:00, 159.60s/it]

top_n      Oracle Recall
5                 0.9214
10                0.9535
20                0.9676
30                0.9689
50                0.9701


## Bước 10 — So sánh Recall/Precision: có rerank vs không rerank (đúng công thức scoring.py, luôn 5 đáp án)

In [ ]:
# predictions_no_rerank = predict_questions_recall_first(
#     questions_sample, alpha=ALPHA, hybrid_top_n=HYBRID_TOP_N, use_rerank=False
# )
# predictions_with_rerank = predict_questions_recall_first(
#     questions_sample, alpha=ALPHA, hybrid_top_n=HYBRID_TOP_N, use_rerank=True, rerank_batch_size=128
# )

# def summarize(predictions, golds, label):
#     p_sum, r_sum = 0.0, 0.0
#     for pred, gold in zip(predictions, golds):
#         p, r = precision_recall(pred, gold)
#         p_sum += p
#         r_sum += r
#     n = len(predictions)
#     print(f"{label}: Precision={p_sum/n:.4f}, Recall={r_sum/n:.4f}")

# summarize(predictions_no_rerank, golds_sample, "KHÔNG rerank")
# summarize(predictions_with_rerank, golds_sample, "CÓ rerank")

## Bước 11 — Áp dụng cho tập test thật (`public-official.json`) và xuất file submit

In [15]:
INPUT_PATH = "/kaggle/input/datasets/thnhnguynchtin/legalir/public-official.json"
OUTPUT_PATH = "/kaggle/working/submission.json"
USE_RERANK_FOR_SUBMIT = True   # đổi theo kết quả so sánh ở Bước 10

with open(INPUT_PATH, encoding="utf-8") as f:
    test_data = json.load(f)

test_qids = list(test_data.keys())
test_questions = [test_data[qid]["question"] for qid in test_qids]
print(f"Số câu hỏi cần dự đoán: {len(test_questions)}")

predictions_test = predict_questions_recall_first(
    test_questions, alpha=best_alpha, hybrid_top_n=HYBRID_TOP_N,
    num_answers=NUM_ANSWERS, use_rerank=USE_RERANK_FOR_SUBMIT, rerank_batch_size=128
)

lengths = [len(p) for p in predictions_test]
print(f"Kiểm tra số lượng đáp án: min={min(lengths)}, max={max(lengths)} (phải LUÔN = {NUM_ANSWERS})")

submission = {qid: {"answer": pred} for qid, pred in zip(test_qids, predictions_test)}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print(f"\nĐã lưu file submit: {OUTPUT_PATH}")
for qid in test_qids[:3]:
    print(f"\n[{qid}] {test_data[qid]['question']}")
    print(f"  -> Dự đoán: {submission[qid]['answer']}")

Số câu hỏi cần dự đoán: 1000
Đang dự đoán cho 1000 câu hỏi (alpha=0.75, top_n=30, rerank=True)...


Predicting: 100%|██████████| 10/10 [1:24:03<00:00, 504.32s/it]

Kiểm tra số lượng đáp án: min=1, max=5 (phải LUÔN = 5)

Đã lưu file submit: /kaggle/working/submission.json

[38096] Đề nghị xem xét lại quyết định đình chỉ tiến hành thủ tục phá sản được xem xét, giải quyết trong thời hạn bao nhiêu ngày làm việc?
  -> Dự đoán: ['277391', '198465', '217416', '234443', '170717']

[63410] Người giúp việc cho Giám đốc Nhà khách Dân tộc có trách nhiệm như thế nào?
  -> Dự đoán: ['190389', '88178', '130501', '39278', '173802']

[37834] Nhiệm kỳ của Hội thẩm nhân dân cấp huyện là bao lâu?
  -> Dự đoán: ['211090', '12319', '113956', '125833', '296536']
